In [0]:
%run ./secrets-template

In [0]:
%python
import os

ssh_priv_key = dbutils.secrets.get(scope="brev", key="ssh_private_key")

with open("/tmp/ssh_private_key_4", "w") as f:
    f.write(ssh_priv_key + "\n")
os.chmod("/tmp/ssh_private_key_4", 0o600)

ssh_user = dbutils.secrets.get(scope='brev', key='ssh_user').strip().splitlines()[-1]
os.environ['SSH_USER'] = ssh_user
print(f'SSH_USER: {ssh_user}')

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
export PATH="\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv/uvx
export UV_CACHE_DIR=/ephemeral/cache/uv
# Non-interactive HF login. 'hf auth login' reads the token from stdin, so a pipe
# is unreliable -> use --token. Writes ~/.cache/huggingface/token, which any
# huggingface_hub install (incl. the converter below) then picks up automatically.
uvx --from huggingface_hub hf auth login --token "$HF_TOKEN" --add-to-git-credential
uvx --from huggingface_hub hf auth whoami
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
set -e
export PATH="\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
export UV_CACHE_DIR=/ephemeral/cache/uv
export UV_LINK_MODE=copy

# 1. Make sure the GR00T env is ready
cd \$HOME/Isaac-GR00T
uv sync --python 3.10
.venv/bin/python -c "import gr00t; print('gr00t:', gr00t.__file__)"

# 2. Install LeRobot into GR00T's .venv at the pinned converter commit
if [ ! -d \$HOME/lerobot ]; then
  git clone https://github.com/huggingface/lerobot.git \$HOME/lerobot
fi
cd \$HOME/lerobot
git fetch --all
git checkout f25ac02
\$HOME/Isaac-GR00T/.venv/bin/python -m pip install -e . --no-deps

# Converter dependencies
\$HOME/Isaac-GR00T/.venv/bin/python -m pip install -U huggingface_hub jsonlines pyarrow numpy tqdm

# ffmpeg (video decode for the converter)
sudo apt-get update
sudo apt-get install -y ffmpeg

\$HOME/Isaac-GR00T/.venv/bin/python -c "import lerobot; print('lerobot:', lerobot.__file__)"

# 3+4. Run the converter. It downloads the v3 dataset from HF (repo id
# \$HF_USER/\$DATASET_NAME) and writes the v2.1 dataset under
# /ephemeral/$HF_USER/$DATASET_NAME. Use .venv/bin/python, NOT 'uv run python'.
cd \$HOME/Isaac-GR00T
.venv/bin/python scripts/lerobot_conversion/convert_v3_to_v2.py \\
  --repo-id "$HF_USER/$DATASET_NAME" \\
  --root /ephemeral
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd /ephemeral/lerobot_datasets/merged_dataset/meta/
 curl -sL -o /ephemeral/alex-luci/bi_so_101_clothes/meta/$MODALITY_JSON \
   -H "Authorization: Bearer $DATABRICKS_TOKEN" \
   "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_JSON"
ls -la /ephemeral/alex-luci/bi_so_101_clothes/meta/
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd /ephemeral/lerobot_datasets/merged_dataset/meta/
 curl -sL -o /ephemeral/alex-luci/bi_so_101_clothes/meta/$MODALITY_PY \
   -H "Authorization: Bearer $DATABRICKS_TOKEN" \
   "$DATABRICKS_HOST/api/2.0/fs/files$MODALITY_FILES_PATH/$MODALITY_PY"
ls -la /ephemeral/alex-luci/bi_so_101_clothes/meta/
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
export PATH="\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin"; # uv
export WANDB_API_KEY=$WANDB_API_KEY
uv run wandb login
cd \$HOME/Isaac-GR00T
EOF

In [0]:
%sh
ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd \$HOME/Isaac-GR00T
rm -f \$HOME/TRAINING_DONE \$HOME/TRAINING_FAILED
tmux kill-session -t finetune 2>/dev/null || true
tmux new-session -d -s finetune
tmux send-keys -t finetune 'export WANDB_API_KEY=$WANDB_API_KEY' C-m
tmux send-keys -t finetune 'export WANDB_NAME=gr00t-n1d6-so100' C-m
tmux send-keys -t finetune 'export PATH=\$HOME/.local/bin:/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin' C-m
tmux send-keys -t finetune 'cd \$HOME/Isaac-GR00T && CUDA_VISIBLE_DEVICES=0 uv run python gr00t/experiment/launch_finetune.py --base_model_path /ephemeral/model/checkpoint-90000 --dataset_path /ephemeral/alex-luci/bi_so_101_clothes --modality_config_path /ephemeral/alex-luci/bi_so_101_clothes/meta/$MODALITY_PY --embodiment_tag NEW_EMBODIMENT --num_gpus 1 --output_dir /ephemeral/finetuned-models --save_steps $SAVE_STEPS --max_steps $MAX_STEPS --use-wandb --warmup_ratio 0.05 --weight_decay 1e-5 --learning_rate 1e-4 --global_batch_size 64 --color_jitter_params brightness 0.3 contrast 0.4 saturation 0.5 hue 0.08 --dataloader_num_workers 16 2>&1 | tee \$HOME/finetune.log && touch \$HOME/TRAINING_DONE || touch \$HOME/TRAINING_FAILED' C-m
#tmux send-keys -t finetune 'exit' C-m
EOF

In [0]:
%sh
echo "Waiting for training to complete..."
while true; do
  echo "$(date): Checking status..."
  
  OUTPUT=$(echo '
    echo "=== TMUX SESSIONS ==="
    tmux list-sessions 2>&1 || echo "NO_SESSIONS"
    echo "=== CHECKING FINETUNE ==="
    if tmux has-session -t finetune 2>/dev/null; then
      echo "STATUS_RUNNING"
    else
      echo "STATUS_DONE"
    fi
  ' | ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP 2>&1)
  
  EXIT_CODE=$?
  
  echo "--- SSH Exit Code: $EXIT_CODE ---"
  echo "--- Full Output ---"
  echo "$OUTPUT"
  echo "-------------------"
  
  if [ $EXIT_CODE -ne 0 ]; then
    echo "WARNING: SSH command failed!"
  fi
  
  if echo "$OUTPUT" | grep -q "STATUS_DONE"; then
    echo "Training complete!"
    break
  fi
  
  sleep 60
done

echo "Cleaning up finetuned model files..."

ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << EOF
cd \$HOME/Isaac-GR00T/finetuned_models/$DATASET_NAME/
du -sh .
EOF

In [0]:
%sh
scp -r -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no \
  $SSH_USER@$BREV_IP:/ephemeral/finetuned-models/checkpoint-30000/ \
  /Volumes/workspace/default/trained_models/bi_so101_clothes_250ep/

In [0]:
%sh
set -e

DEST="/Volumes/workspace/default/trained_models/bi_so101_clothes_250ep/"

rsync -avz --progress \
  -e "ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no" \
  "$SSH_USER@$BREV_IP:/ephemeral/finetuned-models/checkpoint-60000" \
  "$DEST/"

In [0]:
# %sh
# ssh -i /tmp/ssh_private_key_4 -o StrictHostKeyChecking=no $SSH_USER@$BREV_IP << 'EOF'
# sudo shutdown -h now
# EOF
